# Fengyun-2E satellite

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from xgboost.callback import EarlyStopping
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

### Load and preprocess the TLE data


In [2]:
# Replace with appropriate path 
df_tles = pd.read_csv('/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/Fengyun-2E.csv', 
                      index_col=0, parse_dates=True)

# Check if datetime index is timezone-naive or timezone-aware
if df_tles.index.tz is None:
    df_tles.index = df_tles.index.tz_localize('UTC')
else:
    df_tles.index = df_tles.index.tz_convert('UTC')

print(df_tles.describe())

print(df_tles.index.inferred_type)



       eccentricity  argument of perigee  inclination  mean anomaly  \
count   2375.000000          2375.000000  2375.000000   2375.000000   
mean       0.000287             3.185619     0.030617     -3.269659   
std        0.000435             1.075779     0.013323      1.693133   
min        0.000001             0.000002     0.010238     -6.283053   
25%        0.000159             2.589853     0.018576     -4.618659   
50%        0.000252             3.591160     0.030718     -3.424254   
75%        0.000408             3.868018     0.042187     -1.908700   
max        0.020199             6.122826     0.054432     -0.004648   

       Brouwer mean motion  right ascension  
count          2375.000000      2375.000000  
mean              0.004375         1.548779  
std               0.000001         1.561387  
min               0.004367         0.010795  
25%               0.004375         0.959682  
50%               0.004375         1.043691  
75%               0.004375         1.1

### Extract, linear interpolate and scale the Brouwer mean motion

Using only the Brouwer mean motion element and applying linear interpolation on the data point from December 24, 2015. This point was interpolated because it deviated significantly from the surrounding data and was likely an outlier.

In [3]:
df_element_1 = df_tles[["Brouwer mean motion"]].copy()
df_element_1.loc['2015-12-24'] = np.nan
df_element_1 = df_element_1.interpolate(method='linear')
df_element_1 = (df_element_1 - df_element_1.mean())*1e7
df_element_1

,Brouwer mean motion
2011-03-13 19:49:07.927679+00:00,-3.746983
2011-03-14 06:58:21.993311+00:00,-3.840788
2011-03-15 16:04:54.898175+00:00,-4.113038
2011-03-16 22:22:53.164991+00:00,-4.363474
2011-03-17 17:43:47.943840+00:00,-4.500909
...,...
2018-10-17 09:33:42.700896+00:00,6.270388
2018-10-18 12:58:51.403007+00:00,6.180085
2018-10-19 13:06:12.456863+00:00,6.084981
2018-10-20 13:13:28.279199+00:00,6.001222


In [4]:
df_element_1.describe()

,Brouwer mean motion
count,2.375000e+03
mean,1.453468e-12
std,8.292159e+00
min,-8.234707e+01
25%,-1.489873e+00
50%,7.039422e-01
75%,3.030207e+00
max,8.277548e+00


### visualizing the scaled element

In [5]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_element_1.index, 
    y=df_element_1[df_element_1.columns[0]], 
    mode="lines",  
    name="Orbital Element"
))

fig.update_layout(
    title="Brouwer mean motion Over Time",
    xaxis_title="Time",
    yaxis_title="Brouwer mean motion",
    template="plotly",
    width=1400
)

fig.show()


Note: The period from June 3 to June 28 is also highly skewed, but it is not treated as an outlier since it contains multiple consecutive points. These data points are retained as-is and not interpolated.


In [6]:
filtered_df = df_element_1.loc['2015-06-03':'2015-06-28']
#filtered_df = df_element_1.loc['2015-12-24']
filtered_df

,Brouwer mean motion
2015-06-03 22:44:09.786336+00:00,-22.529480
2015-06-05 00:38:13.195104+00:00,-79.832419
2015-06-06 20:28:04.109376+00:00,-79.155714
2015-06-07 20:45:06.208415+00:00,-79.315827
2015-06-08 21:05:21.660288+00:00,-79.521752
2015-06-09 21:21:24.367104+00:00,-79.725495
2015-06-10 21:29:31.547328+00:00,-79.947998
2015-06-11 20:18:36.775871+00:00,-80.153487
2015-06-12 20:44:09.994847+00:00,-80.321018
2015-06-13 01:26:00.755808+00:00,-80.363337


### Create lag features for time series forecasting

In [7]:
NUM_LAG_FEATURES = 3

df_y = df_element_1.copy()
df_x = df_element_1.shift(1).rename(columns={"Brouwer mean motion": "bmm_lag_1"})

for lag in range(2, NUM_LAG_FEATURES + 1):
    df_x[f"bmm_lag_{lag}"] = df_element_1.shift(lag)

# Drop rows with NaNs
df_x = df_x.iloc[NUM_LAG_FEATURES:]
df_y = df_y.iloc[NUM_LAG_FEATURES:]


### Split for hyperparameter tuning


In [8]:
# Split for tuning
split_index = int(len(df_x) * 0.8)
split_date = df_x.index[split_index].strftime("%Y-%m-%d")

df_x_train = df_x[:split_date]
df_y_train = df_y[:split_date]
df_x_test = df_x[split_date:]
df_y_test = df_y[split_date:]

# Train model with early stopping to find best n_estimators
tuned_model = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=10,
    random_state=42
)

tuned_model.fit(
    df_x_train,
    df_y_train.values.ravel(),
    eval_set=[(df_x_train, df_y_train), (df_x_test, df_y_test)],
    verbose=True
)

# View best iteration and RMSE
best_n_estimators = tuned_model.best_iteration + 1  
print(f"Best number of trees: {best_n_estimators}")


[0]	validation_0-rmse:8.35913	validation_1-rmse:2.43097
[1]	validation_0-rmse:7.62291	validation_1-rmse:2.23176
[2]	validation_0-rmse:6.96333	validation_1-rmse:2.05598
[3]	validation_0-rmse:6.37335	validation_1-rmse:1.89682
[4]	validation_0-rmse:5.84700	validation_1-rmse:1.75451
[5]	validation_0-rmse:5.37828	validation_1-rmse:1.63540
[6]	validation_0-rmse:4.96225	validation_1-rmse:1.53264
[7]	validation_0-rmse:4.59421	validation_1-rmse:1.43722
[8]	validation_0-rmse:4.26985	validation_1-rmse:1.35587
[9]	validation_0-rmse:3.98414	validation_1-rmse:1.28721
[10]	validation_0-rmse:3.73328	validation_1-rmse:1.22476
[11]	validation_0-rmse:3.51293	validation_1-rmse:1.17210
[12]	validation_0-rmse:3.32074	validation_1-rmse:1.12995
[13]	validation_0-rmse:3.15366	validation_1-rmse:1.09215
[14]	validation_0-rmse:3.00946	validation_1-rmse:1.06165
[15]	validation_0-rmse:2.88503	validation_1-rmse:1.03347
[16]	validation_0-rmse:2.77888	validation_1-rmse:1.01221
[17]	validation_0-rmse:2.68788	validation


### Retrain final model on full dataset

In [9]:
# Use full data for final model
df_x_full = df_x.copy()
df_y_full = df_y.copy()

final_model = XGBRegressor(
    n_estimators=best_n_estimators,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42
)

final_model.fit(df_x_full, df_y_full.values.ravel())


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=54,
             n_jobs=None, num_parallel_tree=None, ...)

### Make predictions on full data and compute residuals

In [10]:
# Predict and calculate residuals
y_pred_full = final_model.predict(df_x_full)
residuals_full = y_pred_full - df_y_full["Brouwer mean motion"].values

df_result = df_y_full.copy()
df_result["predicted"] = y_pred_full
df_result["residuals"] = residuals_full


#### Plot Observed vs Predicted Values

In [11]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines',
    name='Observed'
))

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines',
    name='Predicted'
))

fig.update_layout(
    title='Observed vs Predicted Brouwer Mean Motion',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    template='plotly_white',
    width= 1200
)

fig.show()


#### Plot Residuals Over Time

In [12]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals'
))

fig.add_hline(y=0, line_dash="dot", line_color="black")

fig.update_layout(
    title='Residuals Over Time',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width= 1400
)

fig.show()


### Plot Residuals with Ground Truth Maneuvers

In [13]:
# Load maneuver data
ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_FENGYUN-2E.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)

# Filter to full range
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual range
y_min = df_result["residuals"].min()
y_max = df_result["residuals"].max()

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)

# Ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    hover_text = f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[y_min, y_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[hover_text, hover_text],
        showlegend=False
    ))

fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))

fig.update_layout(
    title='Residuals with Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()


#### Detect Anomalies (Using 3σ Rule)

In [14]:
import numpy as np

# Calculate mean and standard deviation of residuals
residual_mean = df_result["residuals"].mean()
residual_std = df_result["residuals"].std()
 
# Define upper and lower thresholds
upper = residual_mean + 2* residual_std
lower = residual_mean - 2 * residual_std

# Flag anomalies: residuals that fall outside the threshold range
df_result["anomaly"] = (df_result["residuals"] > upper) | (df_result["residuals"] < lower)

print(f"Anomaly threshold range: {lower:.4f} to {upper:.4f}")
print(" Anomaly timestamps:")
print(df_result[df_result["anomaly"]].index)


Anomaly threshold range: -4.1411 to 4.1417
 Anomaly timestamps:
DatetimeIndex(['2011-03-18 13:33:04.434624+00:00',
               '2011-05-08 21:26:42.042624+00:00',
               '2011-06-22 18:44:25.860768+00:00',
               '2011-08-12 14:42:35.221823+00:00',
               '2011-08-13 20:00:01.840031+00:00',
               '2011-08-14 23:36:47.971584+00:00',
               '2011-08-16 14:31:24.178080+00:00',
               '2011-09-27 06:01:44.510303+00:00',
               '2011-11-18 04:43:23.132927+00:00',
               '2012-02-27 23:31:27.419807+00:00',
               '2012-03-01 06:33:39.019392+00:00',
               '2012-06-03 19:06:40.533408+00:00',
               '2012-07-20 05:43:33.076127+00:00',
               '2012-09-05 14:49:52.375584+00:00',
               '2012-10-25 18:51:51.276096+00:00',
               '2012-11-01 16:33:20.510207+00:00',
               '2012-12-13 17:52:52.836672+00:00',
               '2013-03-16 19:07:08.596128+00:00',
               '20

#### Plotting detected anomalies vs ground truth

In [15]:

# Load ground truth maneuver data
ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_FENGYUN-2E.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)

# Filter maneuver data to match df_result range 
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual y-axis range for drawing vertical maneuver lines 
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines+markers',
    name='Observed',
    marker=dict(size=4),
), secondary_y=False)

# Plot predicted values 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines+markers',
    name='Predicted',
    marker=dict(size=4),
), secondary_y=False)

# Plot residuals 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
), secondary_y=True)

# Plot detected anomalies 
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}',
), secondary_y=True)

# Plot ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left",
    secondary_y=True
)
fig.update_layout(
    title='Observed vs Predicted with Residuals, Detected Anomalies, and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    yaxis2_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=700,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[test_start, test_end]
)

fig.show()



### Residuals with detected anomaly and ground truth maneuver

In [16]:


# Residual range for drawing maneuver lines
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()

#Create figure
fig = go.Figure()

#  Residuals
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    line=dict(color='blue')
))

#  Detected Anomalies
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

#  Ground Truth Maneuver Lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)
fig.update_layout(
    title=dict(
        text='Residuals with Detected Anomalies and Ground Truth Maneuvers',
        font=dict(size=25)  
    ),
    xaxis=dict(
        title='Time(UTC)',
        titlefont=dict(size=25),  
        tickfont=dict(size=22)   
    ),
    yaxis=dict(
        title='Residuals (Observed - Predicted)',
        titlefont=dict(size=25),  
        tickfont=dict(size=22)    
    ),
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        font=dict(size=21),       
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[df_result.index.min(), df_result.index.max()]
)
fig.show()
